# Biohub - Cell Tracking s6_028: 3D-UNet + Transformer + ILP 本番提出パイプライン

- 実行識別子: `MAGIC_STRING = "028-FACTS_ONLY_0990"`
- 成果物接頭語: `RUN_PREFIX = "s6_028-FACTS_ONLY_0990_"`
- 提出ファイル: `/kaggle/working/submission.csv` (Kaggle公式10列フォーマット)
- アーキテクチャ: `TemporalUNet3D` + `SimpleNodeTransformer` + `ILPSolver`
- 本番仕様: `DET_THRESHOLD = 0.99` (原典準拠), `POOL_KERNEL_UM = 5.0`, 単一ベストモデル純化


## 2. パイプラインアーキテクチャ

```mermaid
flowchart TD
    C2[Cell 2: パッケージ導入] --> C3[Cell 3: 定数・設定定義] --> C5[Cell 5: setup_environment]
    C5 --> C6[Cell 6: check_environment] --> C8[Cell 8: 3D-UNet + Transformer 推論]
    C8 --> C10[Cell 10: ILP 大域エネルギー最適化] --> C12[Cell 12: submission.csv 生成] --> C13[Cell 13: main 実行]
```


In [ ]:
# Cell 2: オフライン パッケージライブラリインストール
import os
import sys
import subprocess
from pathlib import Path

if Path("/kaggle/input/biohub-tracking-support-pack-50ep-v1/wheels").exists():
    wheels_dir = Path("/kaggle/input/biohub-tracking-support-pack-50ep-v1/wheels")
elif Path("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/wheels").exists():
    wheels_dir = Path("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/wheels")
else:
    wheels_dir = Path("s6/input/support_pack/wheels") if Path("s6/input/support_pack/wheels").exists() else Path("input/support_pack/wheels")

if wheels_dir.exists():
    print(f"[Cell 2] Installing offline packages from: {wheels_dir.resolve()}")
    offline_pkgs = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "psygnal==0.15.1",
        "imagecodecs==2026.6.26",
    ]
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-deps",
        f"--find-links={wheels_dir}",
        *offline_pkgs
    ], check=True)
    print("[Cell 2] Offline packages installed successfully.")

try:
    import tracksdata
    import pyscipopt
    import ilpy
    import geff
    print("[Cell 2] Required offline packages verified successfully.")
except ImportError as e:
    raise FileNotFoundError(f"致命的エラー: 必須ライブラリのロードに失敗しました: {wheels_dir} ({e})")


In [ ]:
# Cell 3: パラメータ・グローバル変数定義

import os
import sys
import time
import datetime
import math
import copy
from pathlib import Path
import numpy as np
import pandas as pd
import zarr
import torch
from scipy.spatial import KDTree

MAGIC_STRING: str = "028-FACTS_ONLY_0990"
RUN_PREFIX: str = f"s6_{MAGIC_STRING}_"

# 1. 実行モード設定
SUBMIT_TO_COMPETITION: bool = True       # True: 本番提出モード (testデータセットを対象に推論 & submission.csv 生成)
GT_FLG: bool = not SUBMIT_TO_COMPETITION # False: 本番提出時 (GT検証オフ)
GPU_FLG: bool = True                     # True: GPU使用 (CUDA利用可能時), False: 強制CPU

# 2. 対象データセット絞り込み (空リスト [] の場合は見つかった全データセットを実行)
TARGET_DATASETS: list[str] = []          # 本番提出時は全テスト胚を自動走査

# 3. 入力データディレクトリ定義 (確定的な if/elif/else による環境判定)
if SUBMIT_TO_COMPETITION:
    if Path("/kaggle/input/competitions/biohub-cell-tracking-during-development/test").exists():
        DATA_DIR: Path = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development/test")
    elif Path("/kaggle/input/biohub-cell-tracking-during-development/test").exists():
        DATA_DIR: Path = Path("/kaggle/input/biohub-cell-tracking-during-development/test")
    elif Path("/mnt/c/work/aaa/s6/input/test").exists():
        DATA_DIR: Path = Path("/mnt/c/work/aaa/s6/input/test")
    else:
        DATA_DIR: Path = Path("s6/input/test") if Path("s6/input/test").exists() else Path("input/test")
else:
    if Path("/kaggle/input/competitions/biohub-cell-tracking-during-development/train").exists():
        DATA_DIR: Path = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development/train")
    elif Path("/kaggle/input/biohub-cell-tracking-during-development/train").exists():
        DATA_DIR: Path = Path("/kaggle/input/biohub-cell-tracking-during-development/train")
    elif Path("/mnt/c/work/aaa/s6/input/train").exists():
        DATA_DIR: Path = Path("/mnt/c/work/aaa/s6/input/train")
    else:
        DATA_DIR: Path = Path("s6/input/train") if Path("s6/input/train").exists() else Path("input/train")

# 4. サポートパックディレクトリ定義 (確定的な if/elif/else による環境判定)
if Path("/kaggle/input/biohub-tracking-support-pack-50ep-v1").exists():
    SUPPORT_PACK_DIR: Path = Path("/kaggle/input/biohub-tracking-support-pack-50ep-v1")
elif Path("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1").exists():
    SUPPORT_PACK_DIR: Path = Path("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1")
elif Path("/mnt/c/work/aaa/s6/input/support_pack").exists():
    SUPPORT_PACK_DIR: Path = Path("/mnt/c/work/aaa/s6/input/support_pack")
else:
    SUPPORT_PACK_DIR: Path = Path("s6/input/support_pack") if Path("s6/input/support_pack").exists() else Path("input/support_pack")
WHEELS_DIR: Path = SUPPORT_PACK_DIR / "wheels"
WEIGHTS_PATH_PRIMARY: Path = SUPPORT_PACK_DIR / "weights" / "unet_transformer" / "split_0" / "edge_predictor_best.pth"
WEIGHTS_PATH_SECONDARY: Path = SUPPORT_PACK_DIR / "weights" / "unet_transformer" / "split_0" / "checkpoint_last.pth"
REPO_SRC_DIR: Path = SUPPORT_PACK_DIR / "repo" / "src"
REPO_SCRIPTS_DIR: Path = SUPPORT_PACK_DIR / "repo" / "scripts"

# サポートパックのモジュールパスを先行登録
for p in [REPO_SRC_DIR, REPO_SCRIPTS_DIR]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# 5. 深層学習推論 & ILP & パラメータ (028本番確定仕様)
DET_THRESHOLD: float = 0.99           # 3D-UNet 細胞中心確率閾値 (原典設計値 0.99 採用: 局所過学習回避)
DET_TTA: bool = True                 # XYフリップ TTA (Test-Time Augmentation)
POOL_KERNEL_UM: float = 5.0          # 3D極大プーリング抑制半径 (5.0μm config.json準拠)
USE_DUAL_CONSENSUS: bool = False     # 二重モデル無効化 (単一ベストモデル edge_predictor_best.pth に純化)
LOW_MARGIN_THRESHOLD: float = 0.15   # 曖昧領域判定マージン閾値
USE_GAP_CLOSING: bool = False        # 速度外挿ギャップ結合無効化 (誤結合FP増大防止)
GAP_MAX_DT: int = 2                  # ギャップ最大時間差 (dt=2)
GAP_MAX_DIST_UM: float = 8.0         # ギャップ外挿最大許容距離 (μm)

USE_ILP: bool = True                 # ILP大域最適化の有効化
ILP_EDGE_WEIGHT: float = -1.0        # エッジ接続エネルギー重み
ILP_APPEARANCE_WEIGHT: float = 0.1   # 出現ペナルティ
ILP_DISAPPEARANCE_WEIGHT: float = 0.1# 消失ペナルティ
ILP_DIVISION_WEIGHT: float = 1.0     # 分裂イベント許容重み

# 3D異方性物理スケール (Z: 1.625μm, Y: 0.40625μm, X: 0.40625μm)
PHYSICAL_SCALE: tuple[float, float, float] = (1.625, 0.40625, 0.40625)
MATCH_DISTANCE_THRESHOLD_UM: float = 7.0 # GTマッチング距離閾値 (μm)

# 6. GitHub 送信設定 (SUBMITモード時は自動オフ)
PUSH_TO_GITHUB: bool = not SUBMIT_TO_COMPETITION
GITHUB_REPO: str = "https://github.com/kito2718/kaggle_Biohub-Cell_Tracking_During_Development.git"
BRANCH_NAME: str = "main"

if PUSH_TO_GITHUB:
    try:
        from kaggle_secrets import UserSecretsClient
        GITHUB_TOKEN: str = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        import os
        GITHUB_TOKEN: str = os.environ.get("GITHUB_TOKEN", "")
else:
    GITHUB_TOKEN: str = ""

# 7. 出力先設定
OUTPUT_DIR: Path = Path("working")
OUTPUT_SUBMISSION_CSV: str = "submission.csv"
OUTPUT_SUMMARY_CSV: str = f"{RUN_PREFIX}pipeline_summary.csv"
OUTPUT_DETAILS_CSV: str = f"{RUN_PREFIX}pipeline_details.csv"
OUTPUT_EDA_METRICS_XLSX: str = f"{RUN_PREFIX}eda_features_metrics.xlsx"
OUTPUT_EDA_METRICS_CSV: str = f"{RUN_PREFIX}eda_features_metrics.csv"
OUTPUT_EDA_NODE_CSV: str = f"{RUN_PREFIX}eda_node_features.csv"
OUTPUT_EDA_EDGE_CSV: str = f"{RUN_PREFIX}eda_edge_features.csv"
OUTPUT_EDA_TRACK_CSV: str = f"{RUN_PREFIX}eda_track_features.csv"
OUTPUT_EDA_FRAME_CSV: str = f"{RUN_PREFIX}eda_frame_summary.csv"

# 8. グローバル実行状態保持変数
DEVICE: any = None
MODEL_PRIMARY: any = None
MODEL_SECONDARY: any = None
WINDOW_SIZE: int = 2
DOWNSAMPLE: tuple[int, ...] = (1, 4, 4)
DATASET_NAMES: list[str] = []


In [ ]:
# Cell 4: 共通関数定義 (push_to_github & sync_from_github)

def push_to_github(file_name: str, commit_message: str = "Update results") -> bool:
    """成果物ファイルを GitHub リポジトリへ送信する。"""
    if not PUSH_TO_GITHUB or not GITHUB_TOKEN:
        return False
    import subprocess
    import shutil
    import tempfile
    try:
        src_path = OUTPUT_DIR / file_name
        if not src_path.exists():
            return False
        with tempfile.TemporaryDirectory() as tmpdir:
            repo_url = GITHUB_REPO.replace("https://", f"https://oauth2:{GITHUB_TOKEN}@")
            subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH_NAME, repo_url, tmpdir], check=True, capture_output=True)
            dst = Path(tmpdir) / "working" / file_name
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_path, dst)
            subprocess.run(["git", "-C", tmpdir, "config", "user.name", "Kaggle-Notebook"], check=True)
            subprocess.run(["git", "-C", tmpdir, "config", "user.email", "kaggle@example.com"], check=True)
            subprocess.run(["git", "-C", tmpdir, "add", f"working/{file_name}"], check=True)
            res = subprocess.run(["git", "-C", tmpdir, "commit", "-m", commit_message], capture_output=True, text=True)
            if "nothing to commit" in res.stdout or "nothing to commit" in res.stderr:
                return True
            subprocess.run(["git", "-C", tmpdir, "push", "origin", BRANCH_NAME], check=True, capture_output=True)
            print(f"  - [OK] GitHub push 完了: {file_name}")
            return True
    except Exception as e:
        print(f"  - [WARN] GitHub push 失敗 ({file_name}): {e}")
        return False


In [ ]:
# Cell 5: 実行環境セットアップ (setup_environment)

def setup_environment() -> None:
    """Cell 5: デバイス設定 (GPU_FLG連動) および二重モデル (Primary & Secondary) のロードを行う。"""
    global DEVICE, MODEL_PRIMARY, MODEL_SECONDARY, WINDOW_SIZE, DOWNSAMPLE
    import torch

    print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] >>> Cell 5: setup_environment 開始")

    # 1. デバイス選択 (GPU_FLG 連動)
    if GPU_FLG and torch.cuda.is_available():
        DEVICE = torch.device("cuda")
        print(f"  - [DEVICE] GPU 有効化: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB)")
    else:
        DEVICE = torch.device("cpu")
        reason = "GPU_FLG=False (意図的CPUモード)" if not GPU_FLG else "CUDA利用不可"
        print(f"  - [DEVICE] CPU モード稼働 ({reason})")

    # 2. モデル1 (Primary: edge_predictor_best.pth) のロード
    if not WEIGHTS_PATH_PRIMARY.exists():
        raise FileNotFoundError(f"致命的エラー: Primary 重みファイルが存在しません: {WEIGHTS_PATH_PRIMARY.resolve()}")

    from predict_unet_transformer import load_model
    MODEL_PRIMARY, WINDOW_SIZE, DOWNSAMPLE = load_model(WEIGHTS_PATH_PRIMARY, DEVICE)
    print(f"  - [MODEL 1] Primary ロード完了: {WEIGHTS_PATH_PRIMARY.name} -> {DEVICE} (window_size={WINDOW_SIZE}, downsample={DOWNSAMPLE})")

    # 3. モデル2 (Secondary: checkpoint_last.pth) のロード
    if USE_DUAL_CONSENSUS and WEIGHTS_PATH_SECONDARY.exists():
        MODEL_SECONDARY = copy.deepcopy(MODEL_PRIMARY)
        ckpt2 = torch.load(WEIGHTS_PATH_SECONDARY, map_location=DEVICE, weights_only=False)
        state_dict2 = ckpt2.get("model_state_dict", ckpt2)
        MODEL_SECONDARY.load_state_dict(state_dict2)
        MODEL_SECONDARY.to(DEVICE)
        MODEL_SECONDARY.eval()
        print(f"  - [MODEL 2] Secondary ロード完了: {WEIGHTS_PATH_SECONDARY.name} (Epoch: {ckpt2.get('epoch', 'N/A')}, Score: {ckpt2.get('best_score', 'N/A'):.4f})")
    else:
        MODEL_SECONDARY = None
        print(f"  - [MODEL 2] Secondary モデル無効化 (単一モデルモード)")

    print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] <<< Cell 5: setup_environment 正常終了")


In [ ]:
# Cell 6: 実行環境・入力データ検証 (check_environment)

def check_environment() -> None:
    """Cell 6: 実行環境、入力データディレクトリ(DATA_DIR)、データセット一覧を一意に検証しグローバル変数に設定。"""
    global DATASET_NAMES, DATA_DIR
    import psutil
    print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] >>> Cell 6: check_environment 開始")
    print(f"  - [MODE]    SUBMIT_TO_COMPETITION: {SUBMIT_TO_COMPETITION} (GT_FLG: {GT_FLG})")
    print(f"  - [EXEC_ID] MAGIC_STRING: {MAGIC_STRING}")
    print(f"  - [SYSTEM]  CPU: {psutil.cpu_count(logical=True)} cores | RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB")

    if not DATA_DIR.exists():
        raise FileNotFoundError(f"致命的エラー: 入力データディレクトリが存在しません: {DATA_DIR.resolve()}")

    zarr_files = sorted(DATA_DIR.glob("*.zarr"))
    geff_files = sorted(DATA_DIR.glob("*.geff"))

    if TARGET_DATASETS:
        zarr_files = [p for p in zarr_files if p.stem in TARGET_DATASETS]
        geff_files = [p for p in geff_files if p.stem in TARGET_DATASETS]
        print(f"  - [FILTER]  TARGET_DATASETS 指定により {len(zarr_files)} 件に絞り込み: {[p.stem for p in zarr_files]}")

    DATASET_NAMES = [p.name for p in zarr_files]
    print(f"  - [DATA]    DATA_DIR: {DATA_DIR.resolve()}")
    print(f"              .zarr: {len(zarr_files)} 件 | .geff: {len(geff_files)} 件")

    if len(zarr_files) == 0:
        raise FileNotFoundError(f"致命的エラー: DATA_DIR 内に対象の *.zarr が見つかりません: {DATA_DIR.resolve()}")

    if GT_FLG and len(geff_files) == 0:
        raise FileNotFoundError(f"致命的エラー: GT検証モードですが DATA_DIR 内に *.geff が見つかりません: {DATA_DIR.resolve()}")

    print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] <<< Cell 6: check_environment 正常終了")


In [ ]:
# Cell 7: GTデータ読み込み (load_gt_data)

def load_gt_data() -> dict[str, dict]:
    """Cell 7: GT検証モード時にグラウンドトゥルース (.geff) を読み込む。"""
    if not GT_FLG:
        print("[Cell 7] SUBMIT mode: Skipping GT loading.")
        return {}

    print(f"[Cell 7] Loading ground truth tracks for {len(DATASET_NAMES)} datasets...")
    from biohub_tracking.io import open_dataset
    gt_dict = {}

    for name in DATASET_NAMES:
        ds_stem = name.replace(".zarr", "")
        geff_path = DATA_DIR / f"{ds_stem}.geff"
        zarr_path = DATA_DIR / name

        if geff_path.exists():
            zg = zarr.open_group(str(geff_path), mode="r")
            t = zg["nodes"]["props"]["t"]["values"][:]
            z = zg["nodes"]["props"]["z"]["values"][:]
            y = zg["nodes"]["props"]["y"]["values"][:]
            x = zg["nodes"]["props"]["x"]["values"][:]
            if "ids" in zg["nodes"]:
                node_ids = zg["nodes"]["ids"][:]
            elif "id" in zg["nodes"]:
                node_ids = zg["nodes"]["id"][:]
            else:
                node_ids = np.arange(len(t))

            gt_nodes = pd.DataFrame({"node_id": node_ids, "t": t, "z": z, "y": y, "x": x})

            if "ids" in zg["edges"] and zg["edges"]["ids"].shape[0] > 0:
                edges_arr = zg["edges"]["ids"][:]
                edge_s, edge_t = edges_arr[:, 0], edges_arr[:, 1]
            elif "source" in zg["edges"] and "target" in zg["edges"]:
                edge_s, edge_t = zg["edges"]["source"][:], zg["edges"]["target"][:]
            else:
                edge_s, edge_t = [], []

            gt_edges = pd.DataFrame({"source_id": edge_s, "target_id": edge_t})

            meta = zg.attrs.asdict() if hasattr(zg.attrs, "asdict") else dict(zg.attrs)
            geff_meta = meta.get("geff", {}) if isinstance(meta, dict) else {}
            extra = geff_meta.get("extra", {}) if isinstance(geff_meta, dict) else {}
            est_nodes = int(extra.get("estimated_number_of_nodes", meta.get("estimated_number_of_nodes", len(gt_nodes))))

            ds_obj = open_dataset(zarr_path, require_tracks=True, load_image=False)

            gt_dict[ds_stem] = {
                "nodes": gt_nodes,
                "edges": gt_edges,
                "estimated_number_of_nodes": est_nodes,
                "tracks": ds_obj.tracks,
                "geff_path": geff_path,
            }

    print(f"  - Loaded GT for {len(gt_dict)} datasets.")
    return gt_dict


In [ ]:
# Cell 8: 深層学習推論 — 3D-UNet + Transformer 細胞検出・エッジ推論 (detect_nodes_and_edges)

import time
import torch
import numpy as np

def detect_nodes_and_edges(
    max_frames: int | None = None,
) -> dict[str, tuple[np.ndarray, list[tuple[int, int, float, float]]]]:
    """Cell 8: 各データセットに対し、3D-UNetによる細胞中心検出とTransformerによるエッジ推論を実行する。"""
    from predict_unet_transformer import PredictConfig, predict_video
    print(f"[Cell 8] Running deep learning inference on {len(DATASET_NAMES)} datasets...")
    cfg = PredictConfig(
        det_threshold=DET_THRESHOLD,
        det_tta=DET_TTA,
        pool_kernel_um=POOL_KERNEL_UM,
        use_ilp=USE_ILP,
        ilp_edge_weight=ILP_EDGE_WEIGHT,
        ilp_appearance_weight=ILP_APPEARANCE_WEIGHT,
        ilp_disappearance_weight=ILP_DISAPPEARANCE_WEIGHT,
        ilp_division_weight=ILP_DIVISION_WEIGHT,
    )

    raw_preds = {}
    for idx, ds_name in enumerate(DATASET_NAMES, 1):
        ds_stem = ds_name.replace(".zarr", "")
        ds_path = DATA_DIR / ds_name
        t0 = time.time()
        print(f"  [{idx}/{len(DATASET_NAMES)}] Predicting {ds_stem} (Pool: {POOL_KERNEL_UM}um, DetTh: {DET_THRESHOLD})...")

        coords, candidate_edges = predict_video(
            model=MODEL_PRIMARY,
            ds_path=ds_path,
            device=DEVICE,
            cfg=cfg,
            window_size=WINDOW_SIZE,
            max_frames=max_frames,
            unet_batch_size=1,
            downsample=DOWNSAMPLE,
        )
        elapsed = time.time() - t0
        print(f"    -> Inferred {len(coords)} nodes, {len(candidate_edges)} candidate edges in {elapsed:.2f}s")
        raw_preds[ds_stem] = (coords, candidate_edges)

    return raw_preds


In [ ]:
# Cell 9: 検出結果チェック ＆ ノード事後採点 (check_nodes)

def check_nodes(
    raw_preds: dict[str, tuple[np.ndarray, list]],
    gt_dict: dict[str, dict] | None = None
) -> pd.DataFrame:
    """Cell 9: 検出ノードのサマリ統計を集計し、GTモード時は事後採点を算出する。"""
    print("[Cell 9] Auditing detection results and node metrics...")
    scale = np.array(PHYSICAL_SCALE, dtype=np.float32)
    records = []

    for ds_stem, (coords, edges) in raw_preds.items():
        num_nodes = len(coords)
        unique_t = len(np.unique(coords[:, 0])) if num_nodes > 0 else 0
        nodes_per_frame = round(num_nodes / max(1, unique_t), 2)

        rec = {
            "dataset": ds_stem,
            "pred_nodes": num_nodes,
            "unique_frames": unique_t,
            "nodes_per_frame": nodes_per_frame,
            "candidate_edges": len(edges),
        }

        if GT_FLG and gt_dict and ds_stem in gt_dict:
            gt_info = gt_dict[ds_stem]
            gt_df = gt_info["nodes"]
            est_nodes = gt_info["estimated_number_of_nodes"]

            pred_df = pd.DataFrame(coords, columns=["t", "z", "y", "x"])
            pred_df["node_id"] = np.arange(len(pred_df))

            tp_count = 0
            frames = sorted(list(set(gt_df["t"].unique()).union(set(pred_df["t"].unique()))))

            for t_val in frames:
                gt_t = gt_df[gt_df["t"] == t_val]
                pred_t = pred_df[pred_df["t"] == t_val]
                if gt_t.empty or pred_t.empty:
                    continue

                gt_pts = gt_t[["z", "y", "x"]].values * scale
                pred_pts = pred_t[["z", "y", "x"]].values * scale

                tree = KDTree(pred_pts)
                dists, idxs = tree.query(gt_pts, distance_upper_bound=MATCH_DISTANCE_THRESHOLD_UM)
                matched_pred = set()
                for d, p_idx in zip(dists, idxs):
                    if d <= MATCH_DISTANCE_THRESHOLD_UM and p_idx not in matched_pred:
                        matched_pred.add(p_idx)
                        tp_count += 1

            fn_count = len(gt_df) - tp_count
            fp_count = len(pred_df) - tp_count
            recall = tp_count / (tp_count + fn_count) if (tp_count + fn_count) > 0 else 0.0
            precision = tp_count / (tp_count + fp_count) if (tp_count + fp_count) > 0 else 0.0
            f1 = 2 * recall * precision / (recall + precision) if (recall + precision) > 0 else 0.0
            pe_ratio = num_nodes / est_nodes if est_nodes > 0 else 0.0

            rec.update({
                "gt_nodes": len(gt_df),
                "est_nodes": est_nodes,
                "node_tp": tp_count,
                "node_fp": fp_count,
                "node_fn": fn_count,
                "node_recall": round(float(recall), 4),
                "node_precision": round(float(precision), 4),
                "node_f1": round(float(f1), 4),
                "pre_pe_ratio": round(float(pe_ratio), 4),
            })

        records.append(rec)

    summary_df = pd.DataFrame(records)
    print("=" * 80)
    print("【Cell 9: 検出結果 & ノード評価サマリー】")
    print(summary_df.to_string(index=False))
    print("=" * 80)
    return summary_df


In [ ]:
# Cell 10: グラフ構築 ＋ 速度外挿ギャップ結合 ＋ ILP 大域最適化

def generate_gap_candidate_edges(
    coords: np.ndarray,
    candidate_edges: list[tuple[int, int, float, float]],
    scale: np.ndarray,
) -> list[tuple[int, int, float, float]]:
    """dt=2 の一時的検出欠損に対し、直前フレームの運動ベクトル外挿から ILP 用の追加候補エッジを生成する。"""
    # 既存の candidate_edges から直近の最寄り前向き接続を仮定して速度ベクトルを算出
    fwd_best: dict[int, tuple[float, int]] = {}
    for s_id, t_id, prob, dist in candidate_edges:
        if s_id not in fwd_best or dist < fwd_best[s_id][0]:
            fwd_best[s_id] = (dist, t_id)

    # 各ノードの時刻と座標
    node_t = coords[:, 0].astype(np.int64)
    node_pos = coords[:, 1:].astype(np.float32) * scale

    # 時刻別のノードインデックス
    nodes_by_t: dict[int, list[int]] = {}
    for nid, t_val in enumerate(node_t):
        nodes_by_t.setdefault(int(t_val), []).append(nid)

    gap_candidates = []
    # 各ノードについて、dt=2 の先にあるノードへの外挿候補を探索
    for s_id, (v_dist, mid_id) in fwd_best.items():
        t_s = node_t[s_id]
        t_mid = node_t[mid_id]
        if t_mid - t_s != 1:
            continue

        # s_id -> mid_id の速度ベクトル
        vel = node_pos[mid_id] - node_pos[s_id]
        # mid_id からさらに 1 フレーム進んだ先 (t_mid + 1 = t_s + 2) の予測位置
        projected = node_pos[mid_id] + vel

        target_t = t_mid + 1
        cands_next = nodes_by_t.get(target_t, [])
        if not cands_next:
            continue

        for tgt_id in cands_next:
            d = float(np.linalg.norm(projected - node_pos[tgt_id]))
            if d <= GAP_MAX_DIST_UM:
                # 外挿距離に基づく確率スコア (近いほど高確率、最大0.75としてILPに審査させる)
                gap_prob = max(0.51, 0.75 * (1.0 - d / GAP_MAX_DIST_UM))
                gap_candidates.append((mid_id, tgt_id, gap_prob, d))

    return gap_candidates


def build_graph_and_solve_ilp(
    raw_preds: dict[str, tuple[np.ndarray, list]],
) -> dict[str, any]:
    """Cell 10: 速度外挿ギャップ候補エッジを合流させ、ILPSolver による大域的無矛盾最適化を実行する。"""
    import tracksdata as td
    from predict_unet_transformer import build_graph, suppress_output
    print("[Cell 10] Integrating gap closing candidate edges into ILP optimization...")
    scale = np.array(PHYSICAL_SCALE, dtype=np.float32)
    solved_graphs = {}

    for ds_stem, (coords, candidate_edges) in raw_preds.items():
        t0 = time.time()

        # 処方箋: 速度外挿ギャップエッジを ILP 入力前の candidate_edges に合流させる
        all_cands = list(candidate_edges)
        if USE_GAP_CLOSING:
            gap_cands = generate_gap_candidate_edges(coords, candidate_edges, scale)
            all_cands.extend(gap_cands)
            print(f"  - [{ds_stem}] ギャップ外挿候補エッジ {len(gap_cands)} 本を ILP 入力に追加")

        graph = build_graph(coords, all_cands)
        orig_edges = graph.num_edges()

        if USE_ILP and orig_edges > 0:
            solver = td.solvers.ILPSolver(
                edge_weight=ILP_EDGE_WEIGHT * td.EdgeAttr("edge_prob"),
                appearance_weight=ILP_APPEARANCE_WEIGHT,
                disappearance_weight=ILP_DISAPPEARANCE_WEIGHT,
                division_weight=ILP_DIVISION_WEIGHT,
            )
            with suppress_output():
                graph = solver.solve(graph)
            elapsed = time.time() - t0
            print(f"  - [{ds_stem}] ILP optimized: {orig_edges} -> {graph.num_edges()} edges ({elapsed:.2f}s)")
        else:
            print(f"  - [{ds_stem}] Greedy / pass-through: {graph.num_edges()} edges")

        solved_graphs[ds_stem] = graph

    return solved_graphs


In [ ]:
# Cell 11: トラッキングチェック ＆ エッジ事後採点 (check_edges)

def check_edges(
    solved_graphs: dict[str, any],
    gt_dict: dict[str, dict] | None = None
) -> pd.DataFrame:
    """Cell 11: トラッキング結果を集計し、GTモード時は事後採点を算出する。"""
    import polars as pl
    import tracksdata as td
    from biohub_tracking.io import save_graph
    from biohub_tracking.metrics import evaluate as compute_metric, node_recall

    print("[Cell 11] Checking tracking results and computing evaluation metrics...")
    scale = np.array(PHYSICAL_SCALE, dtype=np.float32)
    records = []

    for ds_stem, graph in solved_graphs.items():
        rec = {
            "dataset": ds_stem,
            "final_nodes": graph.num_nodes(),
            "final_edges": graph.num_edges(),
        }

        if GT_FLG and gt_dict and ds_stem in gt_dict:
            gt_info = gt_dict[ds_stem]
            gt_df_nodes = gt_info["nodes"]
            gt_df_edges = gt_info["edges"]
            est_nodes = gt_info["estimated_number_of_nodes"]
            gt_tracks = gt_info["tracks"]

            import tempfile
            with tempfile.TemporaryDirectory() as tmpdir:
                tmp_geff = Path(tmpdir) / "pred.geff"
                save_graph(graph, tmp_geff)
                pred_res = td.graph.IndexedRXGraph.from_geff(tmp_geff)
                pred_rx = pred_res[0] if isinstance(pred_res, tuple) else pred_res

            er = compute_metric(pred_rx, gt_tracks, scale=PHYSICAL_SCALE)
            n_rec = node_recall(pred_rx, gt_tracks) if graph.num_edges() > 0 else 0.0

            edge_denom = er.edge_tp + er.edge_fp + er.edge_fn
            edge_jaccard = er.edge_tp / edge_denom if edge_denom > 0 else 0.0
            div_denom = er.division_tp + er.division_fp + er.division_fn
            div_jaccard = er.division_tp / div_denom if div_denom > 0 else 0.0
            score = edge_jaccard + 0.1 * div_jaccard

            e_tp = er.edge_tp
            e_fp = er.edge_fp
            e_fn = er.edge_fn
            e_rec = e_tp / (e_tp + e_fn) if (e_tp + e_fn) > 0 else 0.0
            e_prec = e_tp / (e_tp + e_fp) if (e_tp + e_fp) > 0 else 0.0
            e_f1 = 2 * e_rec * e_prec / (e_rec + e_prec) if (e_rec + e_prec) > 0 else 0.0
            final_pe = graph.num_nodes() / est_nodes if est_nodes > 0 else 0.0

            rec.update({
                "gt_edges": len(gt_df_edges),
                "edge_tp": e_tp,
                "edge_fp": e_fp,
                "edge_fn": e_fn,
                "edge_recall": round(float(e_rec), 4),
                "edge_precision": round(float(e_prec), 4),
                "edge_f1": round(float(e_f1), 4),
                "post_node_recall": round(float(n_rec), 4),
                "final_pe_ratio": round(float(final_pe), 4),
                "official_edge_jaccard": round(float(edge_jaccard), 4),
                "official_division_jaccard": round(float(div_jaccard), 4),
                "official_score": round(float(score), 4),
            })

        records.append(rec)

    summary_df = pd.DataFrame(records)
    print("=" * 80)
    print("【Cell 11: トラッキング結果 & エッジ評価サマリー】")
    print(summary_df.to_string(index=False))
    print("=" * 80)
    return summary_df


In [ ]:
# Cell 12: 最終出力 & 提出ファイル生成 (save_and_push_results)

def save_and_push_results(
    solved_graphs: dict[str, any],
    node_summary_df: pd.DataFrame,
    edge_summary_df: pd.DataFrame,
) -> pd.DataFrame:
    """Cell 12: tracksdata グラフから Kaggle公式10列フォーマットの submission.csv を生成・保存する。"""
    import polars as pl
    print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] >>> Cell 12: save_and_push_results 開始")
    sub_parts = []

    for ds_stem, graph in solved_graphs.items():
        nodes_df = graph.node_attrs()
        if "solution" in nodes_df.columns:
            nodes_df = nodes_df.filter(pl.col("solution"))

        if len(nodes_df) > 0:
            df_n = nodes_df.select([
                pl.lit(ds_stem).alias("dataset"),
                pl.lit("node").alias("row_type"),
                pl.col("node_id").cast(pl.Int64),
                pl.col("t").cast(pl.Int64),
                pl.col("z").round().cast(pl.Int64),
                pl.col("y").round().cast(pl.Int64),
                pl.col("x").round().cast(pl.Int64),
                pl.lit(-1).cast(pl.Int64).alias("source_id"),
                pl.lit(-1).cast(pl.Int64).alias("target_id"),
            ]).to_pandas()
            sub_parts.append(df_n)

        edges_df = graph.edge_attrs()
        if "solution" in edges_df.columns:
            edges_df = edges_df.filter(pl.col("solution"))

        if len(edges_df) > 0:
            df_e = edges_df.select([
                pl.lit(ds_stem).alias("dataset"),
                pl.lit("edge").alias("row_type"),
                pl.lit(-1).cast(pl.Int64).alias("node_id"),
                pl.lit(-1).cast(pl.Int64).alias("t"),
                pl.lit(-1).cast(pl.Int64).alias("z"),
                pl.lit(-1).cast(pl.Int64).alias("y"),
                pl.lit(-1).cast(pl.Int64).alias("x"),
                pl.col("source_id").cast(pl.Int64),
                pl.col("target_id").cast(pl.Int64),
            ]).to_pandas()
            sub_parts.append(df_e)

    if sub_parts:
        df_sub = pd.concat(sub_parts, ignore_index=True)
    else:
        df_sub = pd.DataFrame([{
            "dataset": "dummy", "row_type": "node", "node_id": 0,
            "t": 0, "z": 0, "y": 0, "x": 0, "source_id": -1, "target_id": -1
        }])

    df_sub = df_sub.sort_values(
        by=["dataset", "row_type", "t", "node_id"],
        ascending=[True, False, True, True]
    ).reset_index(drop=True)

    df_sub.insert(0, "id", range(len(df_sub)))
    for col in ["id", "node_id", "t", "z", "y", "x", "source_id", "target_id"]:
        df_sub[col] = df_sub[col].astype("int64")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    sub_paths = [Path("submission.csv"), OUTPUT_DIR / OUTPUT_SUBMISSION_CSV]
    for sp in set(sub_paths):
        sp.parent.mkdir(parents=True, exist_ok=True)
        df_sub.to_csv(sp, index=False)
        print(f"  - [OK] 提出用ファイル出力完了: {sp.resolve()} (全 {len(df_sub):,} 行 | ノード: {(df_sub['row_type']=='node').sum():,} 行, エッジ: {(df_sub['row_type']=='edge').sum():,} 行)")

    if GT_FLG and not edge_summary_df.empty:
        details_df = pd.merge(node_summary_df, edge_summary_df, on="dataset", how="outer") if not node_summary_df.empty else edge_summary_df
        details_path = OUTPUT_DIR / OUTPUT_DETAILS_CSV
        details_df.to_csv(details_path, index=False)
        print(f"  - [SAVE] 詳細評価CSV保存完了: {details_path.resolve()}")

        summary_record = {
            "magic_string": MAGIC_STRING,
            "total_datasets": len(details_df),
            "macro_node_recall": round(float(details_df["node_recall"].mean()), 4) if "node_recall" in details_df else 0.0,
            "macro_node_precision": round(float(details_df["node_precision"].mean()), 4) if "node_precision" in details_df else 0.0,
            "macro_node_f1": round(float(details_df["node_f1"].mean()), 4) if "node_f1" in details_df else 0.0,
            "macro_edge_recall": round(float(details_df["edge_recall"].mean()), 4) if "edge_recall" in details_df else 0.0,
            "macro_edge_precision": round(float(details_df["edge_precision"].mean()), 4) if "edge_precision" in details_df else 0.0,
            "macro_edge_f1": round(float(details_df["edge_f1"].mean()), 4) if "edge_f1" in details_df else 0.0,
            "macro_official_jaccard": round(float(details_df["official_edge_jaccard"].mean()), 4) if "official_edge_jaccard" in details_df else 0.0,
            "macro_official_score": round(float(details_df["official_score"].mean()), 4) if "official_score" in details_df else 0.0,
            "macro_final_pe_ratio": round(float(details_df["final_pe_ratio"].mean()), 4) if "final_pe_ratio" in details_df else 0.0,
        }
        summary_df = pd.DataFrame([summary_record])
        summary_path = OUTPUT_DIR / OUTPUT_SUMMARY_CSV
        summary_df.to_csv(summary_path, index=False)
        print(f"  - [SAVE] 全体マクロ評価サマリーCSV保存完了: {summary_path.resolve()}")

        print("=" * 80)
        print("【Cell 12: パイプライン全体達成サマリー】")
        for k, v in summary_record.items():
            print(f"  - {k}: {v}")
        print("=" * 80)

        if PUSH_TO_GITHUB and GITHUB_TOKEN:
            push_to_github(OUTPUT_SUMMARY_CSV, commit_message=f"s6_028: push summary {MAGIC_STRING}")
            push_to_github(OUTPUT_DETAILS_CSV, commit_message=f"s6_028: push details {MAGIC_STRING}")

    return df_sub


In [ ]:
# Cell 13: メイン関数 (main エントリポイント)

def main():
    """Cell 13: 3D-UNet + Dual Transformer + ILP + Gap Closing パイプライン一気通貫実行エントリポイント。"""
    start_total = time.time()
    print("=" * 80)
    print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] >>> Cell 13: main パイプライン開始")
    print(f"  - 実行識別子: {MAGIC_STRING}")
    print(f"  - 成果物接頭語: {RUN_PREFIX}")
    print(f"  - 実行モード: {'TRAIN / EVAL (GT検証)' if GT_FLG else 'SUBMIT (提出用)'}")
    print(f"  - GPUフラグ:  {'有効 (GPU優先)' if GPU_FLG else '無効 (強制CPU)'}")
    print(f"  - 改善施策: POOL_KERNEL={POOL_KERNEL_UM}um, DET_THRESHOLD={DET_THRESHOLD}, DUAL_CONSENSUS={USE_DUAL_CONSENSUS}, GAP_CLOSING={USE_GAP_CLOSING}")
    print("=" * 80)

    setup_environment()
    check_environment()
    gt_dict = load_gt_data()
    raw_preds = detect_nodes_and_edges()
    node_summary_df = check_nodes(raw_preds, gt_dict)
    solved_graphs = build_graph_and_solve_ilp(raw_preds)
    edge_summary_df = check_edges(solved_graphs, gt_dict)

    if GT_FLG:
        extract_all_features(solved_graphs, raw_preds, gt_dict)

    df_sub = save_and_push_results(solved_graphs, node_summary_df, edge_summary_df)

    elapsed_total = time.time() - start_total
    print("=" * 80)
    print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] <<< Cell 13: main パイプライン全処理完了 (所要時間: {elapsed_total:.2f} 秒)")
    print("=" * 80)


if __name__ == "__main__":
    main()
